---
## Memory-Augmented Agent


Autonomous agents can make decisions and formulate plans, but a critical limitation remains: how can they retain context and learn from past experiences beyond the current interaction? Many foundational agents operate solely in the present. **Memory-Augmented agents** incorporate mechanisms that preserve context over time, significantly improving personalization, coherence, and long-term reasoning.

### Three Types of Memory

| Memory Type | Role | Trigger Condition | Failure Mode if Skipped |
|---|---|---|---|
| **Working** | Short-term session buffer (LLM prompt) | Active session, current turn | Loss of immediate conversational coherence |
| **Episodic** | Historical interactions & events | Cross-session continuity needed | Agent treats each session as new, breaking personalization |
| **Semantic** | Structured facts & world knowledge | Domain knowledge required | Hallucinated or stale answers |

> **📝 Note — Memory Selection Guide **  
> Use this guide to select which memory system to query: **Working memory** when the LLM context window is being populated directly during an active session. **Episodic memory** via vector similarity search over interaction history when cross-session continuity is needed. **Semantic memory** via keyword or semantic search over a knowledge base when factual or domain knowledge is required that isn't answerable from the session alone.

> **📝 Note — Vector Database Options **  
> Practical implementations include: **Chroma** (well-suited for local development and prototyping), **Pinecone** (managed cloud service optimized for production-scale retrieval), and **Weaviate** (open source with hybrid keyword and vector search support).


In [1]:
# Standard library imports
import os
import sys
import time
from datetime import datetime
from dataclasses import dataclass, field
from functools import wraps

# Ensure local modules are importable regardless of working directory
sys.path.insert(0, ".")

# Supporting modules (must exist alongside this notebook)
from color_logger import log_info, log_success, log_error, log_warn, log_section
from resilience import fail_gracefully
from mock_llm import MockLLM, MockVectorDB, MockResponse

log_success("All imports successful")

[SUCCESS 18:28:11] All imports successful


In [2]:
import os
os.environ["LLM_PROVIDER"] = "openai"

# ── Environment Detection & Simulation Mode Toggle ─────────────

# If an API key is found in .env or via getpass, live mode activates.
# Otherwise, Simulation Mode engages the MockLLM and MockVectorDB.

log_section("Environment Detection")

# Attempt to load .env (no-op if file missing)
try:
    from dotenv import load_dotenv
    load_dotenv()
    log_info("Loaded .env file")
except ImportError:
    log_warn("python-dotenv not installed — skipping .env loading")

# Check for API key
api_key = os.getenv("OPENAI_API_KEY", "").strip()

if not api_key:
    try:
        import getpass
        # In non-interactive environments, this will remain empty
        api_key = getpass.getpass(
            "Enter OpenAI API key (or press Enter for Simulation Mode): "
        ).strip()
    except (EOFError, OSError, Exception):
        api_key = ""

# ── Initialize clients based on mode ───────────────────────────
SIMULATION_MODE = not bool(api_key)

if SIMULATION_MODE:
    llm_client = MockLLM()
    vector_db = MockVectorDB()
    log_warn("SIMULATION MODE active — using MockLLM and MockVectorDB")
    log_info("No API key required. All agent demos will run with mock data.")
else:
    # ── Live LLM wrapper matching MockLLM.generate() interface ─
    try:
        from openai import OpenAI as _OpenAI

        class _LiveLLM:
            """Wraps OpenAI GPT-4o to match MockLLM.generate() interface.
            Returns MockResponse-compatible objects so downstream agent
            code works unchanged."""

            _INTENT_KW = {
                "service_outage": ["outage", "down", "not working", "internet", "offline"],
                "billing_issue": ["bill", "charge", "invoice", "payment", "refund"],
                "complex_technical": ["configure", "setup", "integrate", "deploy", "debug"],
                "marketing_campaign_plan": ["marketing", "campaign", "plan", "launch"],
                "healthcare_query": ["health", "medical", "patient", "diagnosis", "clinical"],
            }

            def __init__(self, client, model="gpt-4o"):
                self._client = client
                self._model = model
                self._fallback = MockLLM()

            def _detect_intent(self, prompt):
                lower = prompt.lower()
                for intent, kws in self._INTENT_KW.items():
                    if any(kw in lower for kw in kws):
                        return intent
                return "general_inquiry"

            def generate(self, prompt, **kwargs):
                try:
                    resp = self._client.chat.completions.create(
                        model=self._model,
                        messages=[{"role": "user", "content": prompt}],
                        max_tokens=1024,
                    )
                    content = resp.choices[0].message.content
                    intent = self._detect_intent(prompt)
                    return MockResponse(
                        content=content,
                        intent=intent,
                        confidence=0.90,
                        reasoning_steps=["Live LLM call via OpenAI GPT-4o"],
                        strategy="full_autonomous_resolution",
                        required_tools=[],
                        escalation_risk=0.1,
                        metadata={"provider": "openai", "model": self._model},
                    )
                except Exception as e:
                    log_warn(f"OpenAI API error, falling back to MockLLM: {e}")
                    return self._fallback.generate(prompt, **kwargs)

        _oai = _OpenAI(api_key=api_key)
        llm_client = _LiveLLM(_oai, model="gpt-4o")
        vector_db = MockVectorDB()
        log_success(f"LIVE MODE — OpenAI GPT-4o (key ...{api_key[-4:]})")
        log_info("Real API calls with MockLLM fallback on errors.")
    except Exception as e:
        log_warn(f"Failed to init OpenAI client: {e} — falling back to MockLLM")
        llm_client = MockLLM()
        vector_db = MockVectorDB()
        SIMULATION_MODE = True



────────────────────────────────────────────────────────────
  Environment Detection
────────────────────────────────────────────────────────────

[INFO  18:28:11] Loaded .env file
[INFO  18:28:13] MockVectorDB initialized with 0 pre-seeded episodic records
[SUCCESS 18:28:13] LIVE MODE — OpenAI GPT-4o (key ...SMYA)
[INFO  18:28:13] Real API calls with MockLLM fallback on errors.


In [3]:
# ── MemoryAugmentedAgent Class ─────────────────────────────────
# Memory-Augmented Agent 
#
# Core logic: retrieve relevant episodic memories to provide
# context for the current query, generate a response with the
# LLM, then store the new interaction for future use.

log_section("Memory-Augmented Agent")


class MemoryAugmentedAgent:
    """Agent with episodic memory for context-aware interactions.

    Retrieves relevant past interactions from a vector database,
    injects them into the LLM prompt as context, generates a
    response, and stores the new interaction for future retrieval.

    """

    def __init__(self, llm_client_ref, vector_db_ref) -> None:
        """Initialize with LLM client and vector database.

        Args:
            llm_client_ref: MockLLM or real LLM client.
            vector_db_ref: MockVectorDB or real vector DB.
        """
        self.llm = llm_client_ref
        self.episodic_memory = vector_db_ref
        self.interaction_count: int = 0

    @fail_gracefully(
        fallback_value="I apologize, but I was unable to process your request.",
    )
    def process_interaction(
        self, user_id: str, user_query: str
    ) -> str:
        """Process a user query with episodic memory augmentation.

        Steps:
        1. Retrieve relevant past interactions from episodic memory
        2. Construct a prompt that includes retrieved memories
        3. Generate a response via the LLM
        4. Store the current interaction for future use

        Args:
            user_id: Unique user identifier.
            user_query: Current user query text.

        Returns:
            Generated response string.

        """
        self.interaction_count += 1
        log_info(f"Interaction #{self.interaction_count} for user '{user_id}'")

        # 1. Retrieve relevant past interactions (Ref: p. 141-26)
        log_info("Step 1: Retrieving episodic memories...")
        relevant_memories = self.episodic_memory.search(
            query=user_query,
            user_id=user_id,
            limit=3,
        )

        if relevant_memories:
            log_success(f"Retrieved {len(relevant_memories)} relevant memories:")
            for i, mem in enumerate(relevant_memories):
                log_info(
                    f"  Memory {i+1}: '{mem['query'][:50]}...' "
                    f"(relevance: {mem.get('relevance_score', 'N/A')})"
                )
        else:
            log_warn("No relevant memories found — first interaction")

        # 2. Construct prompt with memory context
        memory_context = self._format_memories(relevant_memories)
        prompt = (
            f"You are a personalized healthcare assistant.\n"
            f"Here is the user's current query: \"{user_query}\"\n"
            f"For context, here are relevant past interactions "
            f"with this user:\n{memory_context}\n"
            f"Provide a helpful and context-aware response."
        )
        log_info("Step 2: Prompt constructed with memory context")

        # 3. Generate response
        log_info("Step 3: Generating response...")
        #response = self.llm.generate(user_query)
        response = self.llm.generate(prompt)
        response_text = response.content

        # 4. Store interaction in episodic memory (Ref: p. 138)
        log_info("Step 4: Storing interaction in episodic memory...")
        interaction_record = {
            "user_id": user_id,
            "query": user_query,
            "response_summary": response_text[:100],
            "timestamp": datetime.now().isoformat(),
        }
        self.episodic_memory.add(interaction_record)

        log_success(f"Interaction #{self.interaction_count} complete")
        return response_text

    def _format_memories(self, memories: list[dict]) -> str:
        """Format retrieved memories for prompt injection.

        Args:
            memories: List of memory dicts from vector DB search.

        Returns:
            Formatted string for inclusion in LLM prompt.

        Chapter Reference:
            Section 5.3, prompt construction (p. 139)
        """
        if not memories:
            return "(No prior interactions found)"

        lines = []
        for mem in memories:
            lines.append(
                f"- [{mem.get('timestamp', 'unknown')}] "
                f"Query: '{mem.get('query', '')}' → "
                f"Response: '{mem.get('response_summary', '')}'"
            )
        return "\n".join(lines)

    def get_prompt_preview(
        self, user_id: str, user_query: str
    ) -> str:
        """Show the full prompt that would be sent to the LLM.

        Useful for readers to understand how memory retrieval
        augments generation — without actually calling the LLM.

        Args:
            user_id: User identifier for memory lookup.
            user_query: Current query text.

        Returns:
            The complete prompt string.

        Chapter Reference:
            Section 5.3, Prompt inspection (p. 139)
        """
        memories = self.episodic_memory.search(
            query=user_query, user_id=user_id, limit=3
        )
        memory_context = self._format_memories(memories)
        return (
            f"You are a personalized healthcare assistant.\n"
            f"Here is the user's current query: \"{user_query}\"\n"
            f"For context, here are relevant past interactions "
            f"with this user:\n{memory_context}\n"
            f"Provide a helpful and context-aware response."
        )


log_success("MemoryAugmentedAgent class defined")


────────────────────────────────────────────────────────────
  Memory-Augmented Agent
────────────────────────────────────────────────────────────

[SUCCESS 18:28:13] MemoryAugmentedAgent class defined


In [4]:
# ── Healthcare Case Study: 3-Turn Conversation ─────────────────
#  Case Study: A personalized healthcare assistant 
#
# Turn 1: Patient reports fatigue (new interaction)
# Turn 2: Patient follows up about diet change (retrieves Turn 1)
# Turn 3: Patient mentions iron deficiency (retrieves both priors)

log_section(
    "CASE STUDY: Personalized Healthcare Assistant",
)

# Reset vector DB to start fresh with pre-seeded records
vector_db.reset()

health_agent = MemoryAugmentedAgent(llm_client, vector_db)

# ── Turn 1: Initial fatigue report ─────────────────────────────
log_section("Turn 1: Initial Fatigue Report", chapter_ref="p. 139")
response_1 = health_agent.process_interaction(
    user_id="patient_42",
    user_query="I've been feeling very fatigued lately",
)
log_info(f"Agent response: {response_1}...")

# ── Turn 2: Follow-up about diet change ────────────────────────
print()
log_section("Turn 2: Diet Change Follow-Up")
response_2 = health_agent.process_interaction(
    user_id="patient_42",
    user_query="That diet change you suggested helped a bit, but I'm still tired",
)
log_info(f"Agent response: {response_2}...")

# ── Turn 3: Iron deficiency update ─────────────────────────────
print()
log_section("Turn 3: Iron Deficiency Update")
response_3 = health_agent.process_interaction(
    user_id="patient_42",
    user_query="My allergist said I'm low on iron, what should I do?",
)
log_info(f"Agent response: {response_3}...")

# Summary
print()
log_section("Healthcare Case Study Summary")
log_info(f"Total interactions: {health_agent.interaction_count}")
log_success("Memory accumulation demonstrated across 3 turns")


────────────────────────────────────────────────────────────
  CASE STUDY: Personalized Healthcare Assistant
────────────────────────────────────────────────────────────

[WARN  18:28:13] MockVectorDB: reset to 0 seed records

────────────────────────────────────────────────────────────
  Turn 1: Initial Fatigue Report  [p. 139]
────────────────────────────────────────────────────────────

[INFO  18:28:13] Interaction #1 for user 'patient_42'
[INFO  18:28:13] Step 1: Retrieving episodic memories...
[INFO  18:28:13] MockVectorDB: searched 'I've been feeling very fatigued lately' for user 'patient_42' → 0 results (from 0 user records)
[WARN  18:28:13] No relevant memories found — first interaction
[INFO  18:28:13] Step 2: Prompt constructed with memory context
[INFO  18:28:13] Step 3: Generating response...
[INFO  18:28:17] Step 4: Storing interaction in episodic memory...
[INFO  18:28:17] MockVectorDB: stored record for user 'patient_42' (total records: 1)
[SUCCESS 18:28:17] Interactio

In [5]:
# ── Prompt Inspection ──────────────────────────────────────────
# Prompt construction
#
# Display the actual prompt that would be sent to the LLM for
# Turn 3, showing how retrieved episodic memories are injected
# into the context window.

log_section("Prompt Inspection")

prompt_preview = health_agent.get_prompt_preview(
    user_id="patient_42",
    user_query="My allergist said I'm low on iron, what should I do?",
)

log_info("Full prompt sent to LLM (with injected memories):")
print()
print("=" * 60)
print(prompt_preview)
print("=" * 60)
print()
log_success(
    "This demonstrates how episodic memory retrieval augments "
    "the LLM's context, enabling personalized responses."
)


────────────────────────────────────────────────────────────
  Prompt Inspection
────────────────────────────────────────────────────────────

[INFO  18:28:22] MockVectorDB: searched 'My allergist said I'm low on iron, what should I do?' for user 'patient_42' → 3 results (from 3 user records)
[INFO  18:28:22] Full prompt sent to LLM (with injected memories):

You are a personalized healthcare assistant.
Here is the user's current query: "My allergist said I'm low on iron, what should I do?"
For context, here are relevant past interactions with this user:
- [2026-09-17T18:28:22.736709] Query: 'My allergist said I'm low on iron, what should I do?' → Response: 'It sounds like your allergist has identified that a low iron level might be contributing to your fat'
- [2026-09-17T18:28:17.074930] Query: 'I've been feeling very fatigued lately' → Response: 'I'm sorry to hear that you've been feeling fatigued lately. Fatigue can have many different causes, '
- [2026-09-17T18:28:19.513950] Query